# Proposed Data Cleaning

This notebook creates a **WR-only analysis layer** from the NFL Big Data Bowl 2027 source CSVs. It retains the raw competition files unchanged, validates the cohort and joins, and writes filtered data to `data/processed/wr/`. It does **not** aggregate career success or use post-draft outcomes as pre-draft features.

**Run top to bottom.** The three NFL game-tracking files are excluded by default because they are post-draft and not needed for the pre-draft prediction models; set `INCLUDE_GAME_TRACKING = True` to filter them for descriptive or outcome-side analysis. Large tables are processed in chunks.

In [7]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    if start.is_file():
        start = start.parent
    for path in (start, *start.parents):
        if (path / '.git').exists() or (path / 'pyproject.toml').exists():
            return path
    raise FileNotFoundError(f'No repository root found above {start}. Run within the repository.')

REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / 'data' / 'raw' / 'nfl-big-data-bowl-2027'
if not DATA_DIR.is_dir():
    DATA_DIR = REPO_ROOT / 'data' / 'raw'
OUTPUT_DIR = REPO_ROOT / 'data' / 'processed' / 'wr'

CHUNK_SIZE = 150_000
EXPORT_FILTERED = True
INCLUDE_GAME_TRACKING = True

EXPECTED = [
    'players.csv', 'combine_results.csv', 'combine_tracking.csv',
    'player_career_successes.csv', 'player_play.csv', 'games.csv',
    'game_tracking_2023.csv', 'game_tracking_2024.csv', 'game_tracking_2025.csv',
]
# Find the dataset in nested Kaggle extraction directories, detect collisions.
file_matches = {name: sorted(DATA_DIR.rglob(name)) for name in EXPECTED}
ambiguous = {k: v for k, v in file_matches.items() if len(v) > 1}
if ambiguous:
    raise ValueError('Duplicate CSV filenames found; choose the correct extraction folder: ' + str(ambiguous))
paths = {name: matches[0] for name, matches in file_matches.items() if matches}
assert 'players.csv' in paths, f'players.csv not found in {DATA_DIR}'

display(pd.DataFrame([
    {'Dataset': name, 'Found': name in paths,
     'Size (MiB)': round(paths[name].stat().st_size / 1024**2, 1) if name in paths else np.nan}
    for name in EXPECTED
]))
print('Processed output:', OUTPUT_DIR)

,Dataset,Found,Size (MiB)
0,players.csv,True,0.0
1,combine_results.csv,True,0.0
2,combine_tracking.csv,True,79.9
3,player_career_successes.csv,True,0.0
4,player_play.csv,True,140.1
5,games.csv,True,0.0
6,game_tracking_2023.csv,True,319.9
7,game_tracking_2024.csv,True,684.7
8,game_tracking_2025.csv,True,975.2


Processed output: C:\Users\benca\code\PersonalPythonProjects\BigDataBowl2027\data\processed\wr


## 1. Define and validate the WR cohort

Cohort membership comes **only** from `players.csv`, using `nfl_position == "WR"`. We use normalized numeric `nfl_id` values for consistent joins, without silently dropping malformed identifiers. No NFL career outcomes are used to select the cohort.

In [8]:
players = pd.read_csv(paths['players.csv'], low_memory=False)
required_players = {'nfl_id', 'nfl_position'}
assert required_players.issubset(players.columns), f'Missing players columns: {required_players - set(players.columns)}'

def numeric_id(series, table):
    """Normalize IDs; fail on nonempty values that cannot be represented as integers."""
    numeric = pd.to_numeric(series, errors='coerce')
    bad = series.notna() & numeric.isna()
    fractional = numeric.notna() & numeric.ne(np.floor(numeric))
    if bad.any() or fractional.any():
        raise ValueError(f'{table}: invalid nfl_id values, including {series[bad | fractional].head().tolist()}')
    return numeric.astype('Int64')

players['nfl_id'] = numeric_id(players['nfl_id'], 'players')
assert players['nfl_id'].notna().all(), 'Missing player identifiers in players.csv'
assert not players['nfl_id'].duplicated().any(), 'players.csv must have unique nfl_id values'

wr_players = players.loc[players['nfl_position'].astype('string').str.strip().str.upper().eq('WR')].copy()
assert len(wr_players) > 0, 'No WR records found; inspect nfl_position codes.'
wr_ids = set(wr_players['nfl_id'].astype(int))
print(f'Player records: {len(players):,}; WR cohort: {len(wr_players):,} players')
print('WR draft years:', wr_players['draft_year'].value_counts(dropna=False).sort_index().to_dict() if 'draft_year' in wr_players else 'not available')
display(wr_players.head())

Player records: 510; WR cohort: 106 players
WR draft years: {2023: 38, 2024: 31, 2025: 37}


,nfl_id,display_name,draft_year,nfl_position,birth_date,college_name,college_conference,draft_round,draft_pick_within_round,draft_overall_pick
7,55884,Jaxon Smith-Njigba,2023,WR,2002-02-14,Ohio State,Big Ten Conference,1.0,20.0,20.0
8,55885,Quentin Johnston,2023,WR,2001-09-06,Texas Christian,Big Twelve Conference,1.0,22.0,21.0
9,55887,Jordan Addison,2023,WR,2002-01-27,Southern California,Pacific Twelve Conference,1.0,24.0,23.0
15,55904,Jonathan Mingo,2023,WR,2001-04-20,Mississippi,Southeastern Conference,2.0,8.0,39.0
23,55915,Jayden Reed,2023,WR,2000-04-28,Michigan State,Big Ten Conference,2.0,19.0,50.0


## 2. Column selection for `player_play`

For the initial **career-outcome** analysis, retain IDs, game/play evidence, receiving/rushing production, and a few optional professional performance outcomes. WR rows can represent plays where the player was not targeted; **do not filter to passes or targets**, or player participation and denominators will become biased. `play_description` is retained for inspection but not automatically treated as a reliable touchdown or reception label.

In [9]:
WR_PLAYER_PLAY_COLUMNS = [
    'game_id', 'play_id', 'nfl_id', 'team_abbr', 'lined_up_position',
    'play_description', 'play_nullified_by_penalty', 'pass_result',
    'target', 'rec_yards', 'rush_yards', 'yards_after_catch',
    'fumble', 'fumble_lost', 'route_ran',
    'separation_at_pass_forward', 'cushion', 'expected_yards_after_catch',
]
PLAYER_PLAY_REQUIRED = ['game_id', 'play_id', 'nfl_id']

# Show actual availability without reading all of the source dataset.
if 'player_play.csv' in paths:
    pp_header = pd.read_csv(paths['player_play.csv'], nrows=0).columns.tolist()
    missing_columns = [c for c in WR_PLAYER_PLAY_COLUMNS if c not in pp_header]
    print('Optional documented columns not present:', missing_columns)
    assert set(PLAYER_PLAY_REQUIRED).issubset(pp_header), 'Missing primary player_play join keys'

Optional documented columns not present: []


## 3. Reusable cohort filter with row-count audits

We retain **all available columns** for the Combine and career-success tables so new research directions remain possible. Only `player_play` is narrowed to the explicitly documented subset. Processing is streaming/chunked, and files are saved only under `data/processed/wr/`. Missing expected files are shown in the audit rather than mistaken for empty datasets.

In [10]:
audit_rows = []
processed_paths = {}


def filter_file(filename, *, keep_columns=None, filter_column='nfl_id', allowed_ids=None):
    """Filter CSV in chunks, audit full input, and optionally write filtered CSV."""
    if filename not in paths:
        audit_rows.append({'dataset': filename, 'status': 'MISSING', 'source_rows': None,
                           'wr_rows': None, 'retained_pct': None, 'columns': None})
        return None

    path = paths[filename]
    header = pd.read_csv(path, nrows=0).columns.tolist()
    if filter_column not in header:
        raise KeyError(f'{filename}: required filter column {filter_column!r} not present. Header: {header}')
    if keep_columns is not None:
        required = [filter_column] + (PLAYER_PLAY_REQUIRED if filename == 'player_play.csv' else [])
        absent = set(required) - set(header)
        if absent:
            raise KeyError(f'{filename}: missing required columns: {sorted(absent)}')
        cols = [c for c in keep_columns if c in header]
    else:
        cols = header

    output_path = OUTPUT_DIR / filename
    if EXPORT_FILTERED:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        # Remove stale output if there are zero matching rows this time.
        output_path.unlink(missing_ok=True)
    n_source = n_kept = 0
    invalid_ids = 0
    header_written = False
    for chunk in pd.read_csv(path, chunksize=CHUNK_SIZE, low_memory=False):
        n_source += len(chunk)
        raw_id = chunk[filter_column]
        parsed_id = pd.to_numeric(raw_id, errors='coerce')
        invalid_ids += int((raw_id.notna() & parsed_id.isna()).sum())
        invalid_ids += int((parsed_id.notna() & parsed_id.ne(np.floor(parsed_id))).sum())
        mask = parsed_id.isin(allowed_ids if allowed_ids is not None else wr_ids)
        selected = chunk.loc[mask, cols].copy()
        n_kept += len(selected)
        if EXPORT_FILTERED and not selected.empty:
            selected.to_csv(output_path, mode='a', index=False, header=not header_written)
            header_written = True
    if EXPORT_FILTERED and not header_written:
        pd.DataFrame(columns=cols).to_csv(output_path, index=False)
    audit_rows.append({'dataset': filename, 'status': 'OK', 'source_rows': n_source,
                       'wr_rows': n_kept, 'retained_pct': round(100*n_kept/n_source, 2) if n_source else 0,
                       'columns': len(cols), 'invalid_nonempty_filter_ids': invalid_ids})
    if EXPORT_FILTERED:
        processed_paths[filename] = output_path
    print(f'{filename}: {n_kept:,} / {n_source:,} rows retained ({len(cols)} columns)')
    if invalid_ids:
        print(f'  WARNING: {invalid_ids:,} malformed nonempty {filter_column} values in source.')
    return output_path if EXPORT_FILTERED else None

## 4. Filter pre-draft predictors and NFL career outcomes

`combine_results` and `combine_tracking` provide **pre-draft features**; `player_career_successes` and `player_play` provide **post-draft outcomes**. All are restricted to the same WR cohort. Non-WR records are excluded from outputs.

In [11]:
# Save the canonical cohort, without overwriting the original players.csv.
if EXPORT_FILTERED:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    cohort_path = OUTPUT_DIR / 'players.csv'
    wr_players.to_csv(cohort_path, index=False)
    processed_paths['players.csv'] = cohort_path

audit_rows.append({'dataset': 'players.csv', 'status': 'OK', 'source_rows': len(players),
                   'wr_rows': len(wr_players), 'retained_pct': round(100*len(wr_players)/len(players), 2),
                   'columns': len(wr_players.columns)})

filter_file('combine_results.csv')
filter_file('combine_tracking.csv')
filter_file('player_career_successes.csv')
filter_file('player_play.csv', keep_columns=WR_PLAYER_PLAY_COLUMNS)

combine_results.csv: 106 / 510 rows retained (18 columns)
combine_tracking.csv: 132,378 / 463,189 rows retained (14 columns)
player_career_successes.csv: 106 / 510 rows retained (9 columns)
player_play.csv: 42,806 / 314,197 rows retained (18 columns)


WindowsPath('C:/Users/benca/code/PersonalPythonProjects/BigDataBowl2027/data/processed/wr/player_play.csv')

## 5. Restrict `games` to games with WR observations

`games.csv` contains no `nfl_id`. Retain games referenced by the filtered WR `player_play`, using `game_id`. This is an **outcome-context** table, not a pre-draft predictor.

In [12]:
wr_game_ids = set()
if 'player_play.csv' in processed_paths:
    for chunk in pd.read_csv(processed_paths['player_play.csv'], usecols=['game_id'], chunksize=CHUNK_SIZE):
        wr_game_ids.update(pd.to_numeric(chunk['game_id'], errors='coerce').dropna().astype('int64'))
else:
    print('WR player_play output unavailable; games filtering skipped.')

if wr_game_ids:
    filter_file('games.csv', filter_column='game_id', allowed_ids=wr_game_ids)
else:
    audit_rows.append({'dataset': 'games.csv', 'status': 'SKIPPED: no WR play game IDs',
                       'source_rows': None, 'wr_rows': None, 'retained_pct': None, 'columns': None})

games.csv: 932 / 1,002 rows retained (7 columns)


## 6. Optional NFL in-game tracking filtering

These tables measure performance **after** the draft and must never enter pre-draft predictor features. They can be filtered for separate descriptive analysis or independent outcome validation. Their size can be substantial, so they are disabled by default.

In [13]:
for year in (2023, 2024, 2025):
    name = f'game_tracking_{year}.csv'
    if INCLUDE_GAME_TRACKING:
        filter_file(name)
    else:
        audit_rows.append({'dataset': name, 'status': 'SKIPPED by configuration',
                           'source_rows': None, 'wr_rows': None, 'retained_pct': None, 'columns': None})

game_tracking_2023.csv: 719,353 / 3,665,443 rows retained (12 columns)
game_tracking_2024.csv: 1,158,601 / 7,827,944 rows retained (12 columns)
game_tracking_2025.csv: 1,475,631 / 11,160,903 rows retained (12 columns)


## 7. Audit, sanity checks, and export inventory

The audit distinguishes files that were missing, filtered, or intentionally skipped. Verify unmatched IDs and missing columns before relying on any modeling output. No data is imputed or re-labeled in this notebook.

In [14]:
audit = pd.DataFrame(audit_rows)
display(audit)

if EXPORT_FILTERED:
    audit.to_csv(OUTPUT_DIR / 'cohort_filter_audit.csv', index=False)

    for name in ('combine_results.csv', 'combine_tracking.csv',
                 'player_career_successes.csv', 'player_play.csv'):
        if name not in processed_paths:
            continue
        # Small projections only: confirm that every retained nfl_id is in the cohort.
        for chunk in pd.read_csv(processed_paths[name], usecols=['nfl_id'], chunksize=CHUNK_SIZE):
            observed = pd.to_numeric(chunk['nfl_id'], errors='coerce')
            assert observed.notna().all() and observed.isin(wr_ids).all(), f'{name}: non-WR or invalid IDs in output'

    if 'player_play.csv' in processed_paths:
        seen_keys = set()
        duplicate_keys = 0
        missing_counts = {}
        total_play_rows = 0
        unmatched_game_ids = set()
        known_games = None
        if 'games.csv' in processed_paths:
            known_games = set(pd.to_numeric(
                pd.read_csv(processed_paths['games.csv'], usecols=['game_id'])['game_id'],
                errors='coerce'
            ).dropna().astype('int64'))
        fields = None
        for part in pd.read_csv(processed_paths['player_play.csv'], chunksize=CHUNK_SIZE, low_memory=False):
            total_play_rows += len(part)
            if fields is None:
                fields = [c for c in ('target', 'rec_yards', 'rush_yards', 'yards_after_catch',
                                      'separation_at_pass_forward', 'cushion') if c in part.columns]
            keys = list(zip(part['game_id'], part['play_id'], part['nfl_id']))
            for key in keys:
                if key in seen_keys:
                    duplicate_keys += 1
                else:
                    seen_keys.add(key)
            for field in fields:
                missing_counts[field] = missing_counts.get(field, 0) + int(part[field].isna().sum())
            if known_games is not None:
                observed_games = set(pd.to_numeric(part['game_id'], errors='coerce').dropna().astype('int64'))
                unmatched_game_ids.update(observed_games - known_games)
        print(f'WR player_play duplicate (game_id, play_id, nfl_id) keys: {duplicate_keys:,}')
        if duplicate_keys:
            print('WARNING: inspect duplicate records before summing any outcome.')
        if fields:
            display(pd.DataFrame({'Missing (%)': {
                field: round(100*missing_counts[field]/total_play_rows, 1)
                for field in fields
            }}))
        if known_games is not None:
            print(f'WR game IDs not found in games.csv: {len(unmatched_game_ids):,}')
    print('Saved CSV files:', *[str(p.relative_to(REPO_ROOT)) for p in processed_paths.values()], sep='\n  ')
else:
    print('EXPORT_FILTERED=False: no processed files saved and output validation skipped.')

,dataset,status,source_rows,wr_rows,retained_pct,columns,invalid_nonempty_filter_ids
0,players.csv,OK,510,106,20.78,10,NaN
1,combine_results.csv,OK,510,106,20.78,18,0.0
2,combine_tracking.csv,OK,463189,132378,28.58,14,0.0
3,player_career_successes.csv,OK,510,106,20.78,9,0.0
4,player_play.csv,OK,314197,42806,13.62,18,0.0
5,games.csv,OK,1002,932,93.01,7,0.0
6,game_tracking_2023.csv,OK,3665443,719353,19.63,12,0.0
7,game_tracking_2024.csv,OK,7827944,1158601,14.80,12,0.0
8,game_tracking_2025.csv,OK,11160903,1475631,13.22,12,0.0


WR player_play duplicate (game_id, play_id, nfl_id) keys: 0


,Missing (%)
target,0.0
rec_yards,88.3
rush_yards,100.0
yards_after_catch,88.3
separation_at_pass_forward,12.3
cushion,4.8


WR game IDs not found in games.csv: 0
Saved CSV files:
  data\processed\wr\players.csv
  data\processed\wr\combine_results.csv
  data\processed\wr\combine_tracking.csv
  data\processed\wr\player_career_successes.csv
  data\processed\wr\player_play.csv
  data\processed\wr\games.csv
  data\processed\wr\game_tracking_2023.csv
  data\processed\wr\game_tracking_2024.csv
  data\processed\wr\game_tracking_2025.csv


## Next phase: outcome definitions and the pre-draft modeling table

1. Validate the definition and completeness of `target`, `rec_yards`, touchdowns/receptions, and nullified plays before constructing career production.
2. Build a **player-season outcome table** before aggregating career totals, so seasonal trajectories and unequal follow-up periods can be studied.
3. Join conventional pre-draft features to WR IDs for the baseline.
4. Aggregate the Combine tracking into attempt- and player-level skill metrics separately, then compare baseline vs. augmented models on an identical cohort.

**Guardrail:** Do not use `player_play`, `player_career_successes`, `games`, or `game_tracking_YYYY` as predictor columns for a pre-draft model. Draft-round/pick and derived NGS scores also require an explicit feature-availability decision.